# 第二部分 类与结构体
这一部分要强调一个转变：
**从 C 的“数据 + 操作数据的函数”，过渡到 C++ 的“对象 + 生命周期 + 不变量”**
这里不会着重讲“继承、封装、多态”，对于后续开发 SDK 来说，更重要的是：
```
类
成员函数
构造/析构
访问控制
常量成员函数
this 指针
对象生命周期
堆/栈对象
头文件/实现分离
```

本实验聚焦 `struct` 与 `class` 的基本组织方式。完成后应能够解释：成员字段与成员函数如何组成对象、两者的默认访问权限有何不同，以及工程中应如何选择。

## 实验1：从结构体到类
### 结构体
先从熟悉的结构体入手。下面的 `User` 同时保存状态并提供打印状态的行为。

实验代码：

In [1]:
// 本步骤：引入本实验需要的标准库和公开头文件。
#include <iostream>
#include <string>

In [2]:
// 本步骤：通过代码演示“结构体”并观察结果。
// 定义默认公开成员的 struct，把状态和打印行为放在一起。
struct User {
    std::string name;
    int age;

    void print() {
        std::cout
            << "name = " << name
            << ", age = " << age
            << '\n';
    }
};

// 聚合初始化 User，再调用它提供的成员函数。
{
    User user("BobcGn", 20);
    user.print();
}

name = BobcGn, age = 20


这里已经发生了一个重要的变化：
在 C 语言中：
```C
struct User {
    char name[64];
    int age;
};

void user_print(
    const struct User* user
);
```

本质上是：`数据 + 函数`，二者是分开的


而 C++ 中：
```C++
struct User {
    std::string name;
    int age;

    void print();
};
```
将它们组织成了：
```
User
├── state（状态字段）
│   ├── name
│   └── age
│
└── behavior（行为函数）
    └── print()
```

调用：
```C++
user.print();
```

而不是：
```C++
user_print(&user);
```

---


### 类
将上述代码修改为：

In [ ]:
// 本步骤：通过代码演示“类”并观察结果。
// 定义 class，并显式用 public 暴露与 struct 相同的接口。
class User1 {
public:
    std::string name;
    int age;

    void print() {
        std::cout
            << "name = " << name
            << ", age = " << age
            << '\n';
    }
};

// 初始化 class 对象并调用公开行为，比较两种组织方式。
{
    User1 user1("BobcGn", 20);

    user1.print();
}

上述代码能够正常运行，因为成员前显式写了 `public:`。如果删除 `public:`，`name`、`age` 和 `print()` 都会采用 `class` 的默认私有权限，外部初始化和调用就会产生访问控制错误。

默认访问权限是二者最直观的区别：
- `struct` 的成员默认是 `public`
- `class` 的成员默认是 `private`

除此之外，C++ 的结构体和类在语言能力上基本相同。结构体也可以拥有：
结构体也可以：
- 构造/析构
- 成员函数
- 私有化
- 继承
- 虚函数
- 模板



### 实际工程中的选择：
- 纯数据结构
```C++
struct Point{
    double x;
    double y;
};
```
使用结构体，因为**数据本身就是API**

- 有明显不变量和行为
```C++
class User{
private:
    std::string name_;
    int age_;
public:
    // ...
};
```

使用类，因为**不希望调用者随意破坏内部状态**。调用者只能通过公开成员函数进行受控操作，类可以在这些入口中维持不变量。

这与 SDK 设计关系很密切：公开字段一旦成为 API，后续修改数据布局或约束会更加困难；隐藏实现则为校验、兼容和演进保留空间。

### 实验结论

- `struct` 和 `class` 都能封装状态与行为，选择主要表达设计意图。
- 简单、无不变量的数据载体通常使用 `struct`。
- 需要保护状态、维护不变量或隐藏实现细节时通常使用 `class`。
- `user.print()` 中的成员函数会作用于调用它的那个具体对象。